# Chapter 13 — Intro to Automatic Differentiation: Let's Build a Deep Learning Framework

**Mata Kuliah:** Deep Learning
**Buku:** Grokking Deep Learning (Andrew W. Trask, Manning, 2019)
**Chapter 13:** Intro to Automatic Differentiation - Let's Build A Deep Learning Framework

## Isi Chapter 13

1. Apa itu framework deep learning
2. Tensor
3. Autograd: gradien otomatis lewat computation graph
4. Tensor yang dipakai berkali-kali
5. Menambah operasi: negasi, pengurangan, perkalian, sum, expand, transpose, mm
6. Melatih jaringan dengan autograd, optimizer SGD
7. Layer: Linear, Sequential, loss, nonlinearity
8. Embedding layer dan indexing di autograd
9. Cross entropy dan RNN cell

## Cara membaca notebook ini

Chapter ini membangun sebuah framework mini secara bertahap, dan kelas `Tensor` ditulis
ulang beberapa kali dengan fitur yang bertambah. Saya mengikuti urutan itu: setiap
bagian mendefinisikan ulang `Tensor`, jadi sel harus dijalankan berurutan dari atas.
Hanya bagian terakhir (RNN) yang memakai data, yaitu file bAbI dari repository resmi
buku; sel itu mengunduhnya otomatis. Jalankan di Google Colab.

## 1. Apa itu framework deep learning

Framework seperti PyTorch, TensorFlow, dan Keras mengurangi kode yang berulang. Dua
bagian yang paling bermanfaat adalah backpropagation otomatis dan optimasi otomatis:
cukup menulis forward propagation, dan framework mengurus gradien dan update bobot.
Framework juga memungkinkan training di GPU, yang jauh lebih cepat.

Alasan membangunnya sendiri: jaringan berikutnya (LSTM) kodenya rumit jika ditulis
dengan NumPy murni, karena gradien harus ditulis tangan di banyak tempat. Dengan
membangun framework kecil, saya jadi tahu apa yang dikerjakan framework sungguhan, dan itu
membantu saat men-debug model yang tidak mau belajar.

## 2. Tensor

Tensor adalah versi abstrak dari vektor dan matriks: vektor adalah tensor 1 dimensi,
matriks adalah tensor 2 dimensi, dan seterusnya untuk n dimensi. Versi pertama `Tensor`
hanya menyimpan data NumPy dan mendukung satu operasi, penjumlahan.

In [ ]:
import numpy as np

class Tensor (object):

    def __init__(self, data):
        self.data = np.array(data)

    def __add__(self, other):
        return Tensor(self.data + other.data)

    def __repr__(self):
        return str(self.data.__repr__())

    def __str__(self):
        return str(self.data.__str__())

x = Tensor([1,2,3,4,5])
print(x)

y = x + x
print(y)

Menambah operasi lain tinggal menambah fungsi di kelas ini.

## 3. Autograd: gradien otomatis

Selama ini turunan dihitung tangan untuk setiap jaringan: mulai dari gradien di output,
mundur satu komponen demi satu. Logika itu bisa ditaruh di dalam tensor. Setiap tensor
sekarang punya dua atribut baru:

- `creators`: daftar tensor yang dipakai untuk membuatnya (`None` jika dibuat langsung),
- `creation_op`: nama operasi yang dipakai.

Dengan ini, `z = x + y` otomatis membentuk computation graph: tiga node (`x`, `y`, `z`)
dan dua sisi (`z -> x`, `z -> y`) yang berlabel `add`. Saat `z.backward()` dipanggil,
gradien dikirim ke pembuatnya sesuai operasinya. Untuk penjumlahan, gradien cukup
disalin ke kedua sisi.

In [ ]:
import numpy as np

class Tensor (object):

    def __init__(self, data, creators=None, creation_op = None):
        self.data = np.array(data)
        self.creation_op = creation_op
        self.creators = creators
        self.grad = None

    def backward(self, grad):
        self.grad = grad

        if(self.creation_op == "add"):
            self.creators[0].backward(grad)
            self.creators[1].backward(grad)

    def __add__(self, other):
        return Tensor(self.data + other.data,  creators=[self,other], creation_op="add")

    def __repr__(self):
        return str(self.data.__repr__())

    def __str__(self):
        return str(self.data.__str__())

x = Tensor([1,2,3,4,5])
y = Tensor([2,2,2,2,2])

z = x + y
z.backward(Tensor(np.array([1,1,1,1,1])))

In [ ]:
print(x.grad)
print(y.grad)
print(z.creators)
print(z.creation_op)

Gradien `[1 1 1 1 1]` sampai ke `x` dan `y`, sesuai aturan backpropagation lewat
penjumlahan. Bagian yang menarik: cara ini bekerja secara rekursif, karena setiap tensor
memanggil `.backward()` pada `creators`-nya. Pada graph yang lebih dalam, gradien
mengalir sampai ke daun.

In [ ]:
a = Tensor([1,2,3,4,5])
b = Tensor([2,2,2,2,2])
c = Tensor([5,4,3,2,1])
d = Tensor([-1,-2,-3,-4,-5])

e = a + b
f = c + d
g = e + f

g.backward(Tensor(np.array([1,1,1,1,1])))

print(a.grad)

### Catatan: ini bukan hal baru

Memikirkan gradien yang mengalir di sebuah graph terasa seperti lompatan, tetapi
sebenarnya sama dengan yang sudah dilakukan di chapter RNN: forward ke satu arah, lalu
backpropagation melewati "graph" aktivasi. Bedanya, dulu node dan sisinya tidak disimpan
sebagai struktur data; yang ada hanya list layer dan urutan operasi yang ditulis tangan.
Sekarang antarmuka ini disusun supaya backpropagation tidak perlu ditulis tangan lagi.

Graph yang dibangun saat forward propagation disebut *dynamic computation graph*
karena dibentuk pada saat forward berjalan. Ini dipakai PyTorch dan DyNet. Framework
lama seperti Theano dan TensorFlow versi awal memakai *static graph* yang didefinisikan
sebelum forward berjalan. Dynamic lebih mudah untuk bereksperimen, static biasanya lebih
cepat saat dijalankan. Chapter ini memakai dynamic.

## 4. Tensor yang dipakai berkali-kali

Versi autograd tadi punya bug: gradien hanya bisa dipropagasi ke sebuah tensor sekali.
Kalau tensor yang sama dipakai di beberapa tempat (misalnya bobot jaringan), gradien
dari tiap pemakaian harus dijumlahkan, bukan saling menimpa. Contoh: `b` dipakai dua kali
untuk membuat `f`, jadi gradiennya seharusnya `[2 2 2 2 2]`.

In [ ]:
a = Tensor([1,2,3,4,5])
b = Tensor([2,2,2,2,2])
c = Tensor([5,4,3,2,1])

d = a + b
e = b + c
f = d + e
f.backward(Tensor(np.array([1,1,1,1,1])))

b.grad.data == np.array([2,2,2,2,2])

Hasilnya `False` di semua posisi, karena gradien dari `d` tertimpa oleh gradien dari `e`.

## Memperbaiki autograd untuk tensor multi-pakai

Perbaikannya menambah satu fungsi dan mengubah tiga yang lama:

- `autograd`: penanda apakah tensor ikut dihitung gradiennya,
- `children`: hitungan berapa kali tiap anak (tensor yang dibuat darinya) akan
  mengirim gradien,
- `all_children_grads_accounted_for()`: apakah gradien dari semua anak sudah masuk,
- di `backward`, gradien diakumulasi (`self.grad += grad`) dan tensor menunggu semua
  anaknya sebelum meneruskan gradien ke pembuatnya.

Setiap tensor juga punya `id` acak supaya anak-anaknya bisa dikenali.

In [ ]:
import numpy as np

class Tensor (object):

    def __init__(self,data,
                 autograd=False,
                 creators=None,
                 creation_op=None,
                 id=None):

        self.data = np.array(data)
        self.autograd = autograd
        self.grad = None
        if(id is None):
            self.id = np.random.randint(0,100000)
        else:
            self.id = id

        self.creators = creators
        self.creation_op = creation_op
        self.children = {}

        if(creators is not None):
            for c in creators:
                if(self.id not in c.children):
                    c.children[self.id] = 1
                else:
                    c.children[self.id] += 1

    def all_children_grads_accounted_for(self):
        for id,cnt in self.children.items():
            if(cnt != 0):
                return False
        return True

    def backward(self,grad=None, grad_origin=None):
        if(self.autograd):

            if(grad_origin is not None):
                if(self.children[grad_origin.id] == 0):
                    raise Exception("cannot backprop more than once")
                else:
                    self.children[grad_origin.id] -= 1

            if(self.grad is None):
                self.grad = grad
            else:
                self.grad += grad

            # grads must not have grads of their own
            assert grad.autograd == False

            # only continue backpropping if there's something to
            # backprop into and if all gradients (from children)
            # are accounted for override waiting for children if
            # "backprop" was called on this variable directly
            if(self.creators is not None and
               (self.all_children_grads_accounted_for() or
                grad_origin is None)):

                if(self.creation_op == "add"):
                    self.creators[0].backward(self.grad, self)
                    self.creators[1].backward(self.grad, self)

    def __add__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data + other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="add")
        return Tensor(self.data + other.data)

    def __repr__(self):
        return str(self.data.__repr__())

    def __str__(self):
        return str(self.data.__str__())

a = Tensor([1,2,3,4,5], autograd=True)
b = Tensor([2,2,2,2,2], autograd=True)
c = Tensor([5,4,3,2,1], autograd=True)

d = a + b
e = b + c
f = d + e

f.backward(Tensor(np.array([1,1,1,1,1])))

print(b.grad.data == np.array([2,2,2,2,2]))

Sekarang gradien `b` benar. Hitungan `children` juga mencegah satu anak mengirim gradien
dua kali. Saya coba memanggil `backward` lagi pada `f` untuk melihatnya (percobaan
tambahan, bukan dari kode buku):

In [ ]:
# Percobaan tambahan (bukan dari kode buku): backprop kedua kali
try:
    f.backward(Tensor(np.array([1,1,1,1,1])))
except Exception as err:
    print('Error:', err)

## 5. Bagaimana backpropagation penjumlahan bekerja

Untuk menambah operasi baru, ada dua tempat yang perlu disentuh: fungsi operasinya
(yang membuat tensor baru dengan `creators` dan `creation_op`) dan blok gradiennya di
`.backward()`. Penjumlahan:

```python
def __add__(self, other):
    if(self.autograd and other.autograd):
        return Tensor(self.data + other.data, autograd=True,
                      creators=[self,other], creation_op="add")
    return Tensor(self.data + other.data)
```

Gradiennya: `self.creators[0].backward(self.grad, self)` dan
`self.creators[1].backward(self.grad, self)`. Penjumlahan tidak ditangani di tempat lain,
logika backpropagation umumnya sudah diabstraksikan. Pemanggilan `backward` ke
pembuatnya tetap dilakukan, tetapi baru berjalan jika `autograd` pembuat itu `True`. Dan
tensor hasil punya `autograd=True` hanya jika kedua operandnya `autograd=True`.

### Negasi

Negasi hampir identik dengan penjumlahan, tetapi hanya punya satu pembuat, jadi
`backward` dipanggil sekali dan gradiennya dinegasikan.

In [ ]:
import numpy as np

class Tensor (object):

    def __init__(self,data,
                 autograd=False,
                 creators=None,
                 creation_op=None,
                 id=None):

        self.data = np.array(data)
        self.autograd = autograd
        self.grad = None
        if(id is None):
            self.id = np.random.randint(0,100000)
        else:
            self.id = id

        self.creators = creators
        self.creation_op = creation_op
        self.children = {}

        if(creators is not None):
            for c in creators:
                if(self.id not in c.children):
                    c.children[self.id] = 1
                else:
                    c.children[self.id] += 1

    def all_children_grads_accounted_for(self):
        for id,cnt in self.children.items():
            if(cnt != 0):
                return False
        return True

    def backward(self,grad=None, grad_origin=None):
        if(self.autograd):

            if(grad_origin is not None):
                if(self.children[grad_origin.id] == 0):
                    raise Exception("cannot backprop more than once")
                else:
                    self.children[grad_origin.id] -= 1

            if(self.grad is None):
                self.grad = grad
            else:
                self.grad += grad

            # grads must not have grads of their own
            assert grad.autograd == False

            # only continue backpropping if there's something to
            # backprop into and if all gradients (from children)
            # are accounted for override waiting for children if
            # "backprop" was called on this variable directly
            if(self.creators is not None and
               (self.all_children_grads_accounted_for() or
                grad_origin is None)):

                if(self.creation_op == "add"):
                    self.creators[0].backward(self.grad, self)
                    self.creators[1].backward(self.grad, self)

                if(self.creation_op == "neg"):
                    self.creators[0].backward(self.grad.__neg__())

    def __add__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data + other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="add")
        return Tensor(self.data + other.data)

    def __neg__(self):
        if(self.autograd):
            return Tensor(self.data * -1,
                          autograd=True,
                          creators=[self],
                          creation_op="neg")
        return Tensor(self.data * -1)

    def __repr__(self):
        return str(self.data.__repr__())

    def __str__(self):
        return str(self.data.__str__())

a = Tensor([1,2,3,4,5], autograd=True)
b = Tensor([2,2,2,2,2], autograd=True)
c = Tensor([5,4,3,2,1], autograd=True)

d = a + (-b)
e = (-b) + c
f = d + e

f.backward(Tensor(np.array([1,1,1,1,1])))

print(b.grad.data == np.array([-2,-2,-2,-2,-2]))

Saat `b` dipakai lewat `-b`, gradien yang kembali ikut berbalik tanda, dan sistem umum
backpropagation tidak perlu diubah sama sekali.

## 6. Menambah fungsi lain

Dengan pola yang sama, saya tambahkan pengurangan (`sub`), perkalian elemen (`mul`),
`sum`, `expand`, `transpose`, dan perkalian matriks (`mm`). Turunannya sudah dibahas di
chapter-chapter awal:

- `sub`: gradien disalin ke operand pertama, dan dinegasikan untuk operand kedua.
- `mul`: gradien untuk satu operand adalah gradien dikali operand yang lain.
- `mm`: gradien untuk aktivasi adalah `grad.mm(bobot.transpose())`, dan untuk bobot
  `grad.transpose().mm(aktivasi).transpose()`, sama seperti `layer_1_delta =
  layer_2_delta.dot(weights_1_2.T)` di Chapter 6.
- `sum(dim)` dan `expand(dim, copies)` saling berpasangan: backward dari `sum` adalah
  `expand`, dan backward dari `expand` adalah `sum`.

Di versi ini juga ada kemudahan kecil: kalau `backward()` dipanggil tanpa gradien,
gradiennya otomatis tensor berisi angka 1.

In [ ]:
import numpy as np

class Tensor (object):

    def __init__(self,data,
                 autograd=False,
                 creators=None,
                 creation_op=None,
                 id=None):

        self.data = np.array(data)
        self.autograd = autograd
        self.grad = None
        if(id is None):
            self.id = np.random.randint(0,100000)
        else:
            self.id = id

        self.creators = creators
        self.creation_op = creation_op
        self.children = {}

        if(creators is not None):
            for c in creators:
                if(self.id not in c.children):
                    c.children[self.id] = 1
                else:
                    c.children[self.id] += 1

    def all_children_grads_accounted_for(self):
        for id,cnt in self.children.items():
            if(cnt != 0):
                return False
        return True

    def backward(self,grad=None, grad_origin=None):
        if(self.autograd):

            if(grad is None):
                grad = Tensor(np.ones_like(self.data))

            if(grad_origin is not None):
                if(self.children[grad_origin.id] == 0):
                    raise Exception("cannot backprop more than once")
                else:
                    self.children[grad_origin.id] -= 1

            if(self.grad is None):
                self.grad = grad
            else:
                self.grad += grad

            # grads must not have grads of their own
            assert grad.autograd == False

            # only continue backpropping if there's something to
            # backprop into and if all gradients (from children)
            # are accounted for override waiting for children if
            # "backprop" was called on this variable directly
            if(self.creators is not None and
               (self.all_children_grads_accounted_for() or
                grad_origin is None)):

                if(self.creation_op == "add"):
                    self.creators[0].backward(self.grad, self)
                    self.creators[1].backward(self.grad, self)

                if(self.creation_op == "sub"):
                    self.creators[0].backward(Tensor(self.grad.data), self)
                    self.creators[1].backward(Tensor(self.grad.__neg__().data), self)

                if(self.creation_op == "mul"):
                    new = self.grad * self.creators[1]
                    self.creators[0].backward(new , self)
                    new = self.grad * self.creators[0]
                    self.creators[1].backward(new, self)

                if(self.creation_op == "mm"):
                    c0 = self.creators[0]
                    c1 = self.creators[1]
                    new = self.grad.mm(c1.transpose())
                    c0.backward(new)
                    new = self.grad.transpose().mm(c0).transpose()
                    c1.backward(new)

                if(self.creation_op == "transpose"):
                    self.creators[0].backward(self.grad.transpose())

                if("sum" in self.creation_op):
                    dim = int(self.creation_op.split("_")[1])
                    self.creators[0].backward(self.grad.expand(dim,
                                                               self.creators[0].data.shape[dim]))

                if("expand" in self.creation_op):
                    dim = int(self.creation_op.split("_")[1])
                    self.creators[0].backward(self.grad.sum(dim))

                if(self.creation_op == "neg"):
                    self.creators[0].backward(self.grad.__neg__())

    def __add__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data + other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="add")
        return Tensor(self.data + other.data)

    def __neg__(self):
        if(self.autograd):
            return Tensor(self.data * -1,
                          autograd=True,
                          creators=[self],
                          creation_op="neg")
        return Tensor(self.data * -1)

    def __sub__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data - other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="sub")
        return Tensor(self.data - other.data)

    def __mul__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data * other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="mul")
        return Tensor(self.data * other.data)

    def sum(self, dim):
        if(self.autograd):
            return Tensor(self.data.sum(dim),
                          autograd=True,
                          creators=[self],
                          creation_op="sum_"+str(dim))
        return Tensor(self.data.sum(dim))

    def expand(self, dim,copies):

        trans_cmd = list(range(0,len(self.data.shape)))
        trans_cmd.insert(dim,len(self.data.shape))
        new_data = self.data.repeat(copies).reshape(list(self.data.shape) + [copies]).transpose(trans_cmd)

        if(self.autograd):
            return Tensor(new_data,
                          autograd=True,
                          creators=[self],
                          creation_op="expand_"+str(dim))
        return Tensor(new_data)

    def transpose(self):
        if(self.autograd):
            return Tensor(self.data.transpose(),
                          autograd=True,
                          creators=[self],
                          creation_op="transpose")

        return Tensor(self.data.transpose())

    def mm(self, x):
        if(self.autograd):
            return Tensor(self.data.dot(x.data),
                          autograd=True,
                          creators=[self,x],
                          creation_op="mm")
        return Tensor(self.data.dot(x.data))

    def __repr__(self):
        return str(self.data.__repr__())

    def __str__(self):
        return str(self.data.__str__())

a = Tensor([1,2,3,4,5], autograd=True)
b = Tensor([2,2,2,2,2], autograd=True)
c = Tensor([5,4,3,2,1], autograd=True)

d = a + b
e = b + c
f = d + e

f.backward(Tensor(np.array([1,1,1,1,1])))

print(b.grad.data == np.array([2,2,2,2,2]))

### Catatan tentang `sum` dan `expand`

`sum(dim)` menjumlahkan sepanjang satu dimensi dan menghilangkan dimensi itu.
`expand(dim, copies)` menyalin data sepanjang satu dimensi dan menambah dimensi baru.
Untuk matriks 2 x 3:

In [ ]:
x = Tensor(np.array([[1,2,3],
                     [4,5,6]]))

In [ ]:
x.sum(0)

In [ ]:
x.sum(1)

In [ ]:
x.expand(dim=2, copies=4)

In [ ]:
# Percobaan tambahan (bukan dari kode buku): bentuk hasil expand pada dimensi lain
print(x.data.shape, '->', x.expand(dim=0, copies=4).data.shape)
print(x.data.shape, '->', x.expand(dim=2, copies=4).data.shape)

`sum` mengurangi satu dimensi, `expand` menambahnya. Karena itu `.sum(dim=1)` pada
tensor yang dimensi tersebut berisi empat entri dipasangkan dengan
`.expand(dim=1, copies=4)` saat backpropagation.

## 7. Melatih jaringan dengan autograd

Sebagai pembanding, ini jaringan kecil dengan backpropagation yang ditulis tangan:

In [ ]:
import numpy
np.random.seed(0)

data = np.array([[0,0],[0,1],[1,0],[1,1]])
target = np.array([[0],[1],[0],[1]])

weights_0_1 = np.random.rand(2,3)
weights_1_2 = np.random.rand(3,1)

for i in range(10):

    # Predict
    layer_1 = data.dot(weights_0_1)
    layer_2 = layer_1.dot(weights_1_2)

    # Compare
    diff = (layer_2 - target)
    sqdiff = (diff * diff)
    loss = sqdiff.sum(0) # mean squared error loss

    # Learn: this is the backpropagation piece
    layer_1_grad = diff.dot(weights_1_2.transpose())
    weight_1_2_update = layer_1.transpose().dot(diff)
    weight_0_1_update = data.transpose().dot(layer_1_grad)

    weights_1_2 -= weight_1_2_update * 0.1
    weights_0_1 -= weight_0_1_update * 0.1
    print(loss[0])

Di sini `layer_1`, `layer_2`, dan `diff` harus disimpan sebagai variabel karena dipakai
lagi saat backpropagation, dan setiap gradien harus diteruskan ke matriks bobot yang tepat
secara manual. Dengan autograd, kodenya jauh lebih pendek:

In [ ]:
import numpy
np.random.seed(0)

data = Tensor(np.array([[0,0],[0,1],[1,0],[1,1]]), autograd=True)
target = Tensor(np.array([[0],[1],[0],[1]]), autograd=True)

w = list()
w.append(Tensor(np.random.rand(2,3), autograd=True))
w.append(Tensor(np.random.rand(3,1), autograd=True))

for i in range(10):

    # Predict
    pred = data.mm(w[0]).mm(w[1])

    # Compare
    loss = ((pred - target)*(pred - target)).sum(0)

    # Learn
    loss.backward(Tensor(np.ones_like(loss.data)))

    for w_ in w:
        w_.data -= w_.grad.data * 0.1
        w_.grad.data *= 0

    print(loss)

Kodenya jauh lebih sederhana: tidak ada variabel sementara yang harus disimpan (graph
dinamis yang mengingatnya) dan tidak ada kode backpropagation (ditangani `.backward()`),
jadi peluang salah ketik di bagian gradien juga berkurang.

Loss versi autograd berbeda dari versi tulis tangan di atas (awal 0.58 dibanding 5.07).
Ada dua penyebab, dan saya periksa di sel berikutnya:

1. Setiap `Tensor` mengambil `id` acak dari generator angka acak NumPy yang sama dengan
   yang dipakai untuk bobot. Dengan `seed(0)` yang sama, bobot awal ikut bergeser.
2. Loss berbentuk kuadrat, jadi gradien yang dihitung autograd adalah `2 * diff`,
   sedangkan versi tulis tangan memakai `diff` saja. Langkah efektifnya setengahnya.

In [ ]:
# Percobaan tambahan (bukan dari kode buku): versi manual dengan bobot awal yang sama
# dengan versi autograd, dan langkah update 0.2 (bukan 0.1)
np.random.seed(0)
r = lambda: np.random.randint(0, 100000)    # setiap Tensor baru mengambil id acak

r(); r()                                     # id untuk data dan target
weights_0_1 = np.random.rand(2,3); r()       # bobot pertama, lalu id tensornya
weights_1_2 = np.random.rand(3,1); r()       # bobot kedua, lalu id tensornya

data = np.array([[0,0],[0,1],[1,0],[1,1]])
target = np.array([[0],[1],[0],[1]])

for i in range(10):
    layer_1 = data.dot(weights_0_1)
    layer_2 = layer_1.dot(weights_1_2)
    diff = (layer_2 - target)
    loss = (diff * diff).sum(0)

    layer_1_grad = diff.dot(weights_1_2.transpose())
    weight_1_2_update = layer_1.transpose().dot(diff)
    weight_0_1_update = data.transpose().dot(layer_1_grad)

    weights_1_2 -= weight_1_2_update * 0.2
    weights_0_1 -= weight_0_1_update * 0.2
    print(loss)

Dengan bobot awal yang sama dan langkah 0.2, versi tulis tangan menghasilkan loss yang
sama persis dengan autograd (0.58128304 sampai 0.06849361). Jadi autograd menghitung
gradien yang sama dengan turunan yang saya tulis tangan.

## 8. Optimasi otomatis: SGD

Karena parameter sudah ada di sebuah list, stochastic gradient descent tinggal loop di
akhir. Loop itu saya bungkus menjadi kelas `SGD`.

In [ ]:
class SGD(object):

    def __init__(self, parameters, alpha=0.1):
        self.parameters = parameters
        self.alpha = alpha

    def zero(self):
        for p in self.parameters:
            p.grad.data *= 0

    def step(self, zero=True):

        for p in self.parameters:

            p.data -= p.grad.data * self.alpha

            if(zero):
                p.grad.data *= 0

In [ ]:
import numpy
np.random.seed(0)

data = Tensor(np.array([[0,0],[0,1],[1,0],[1,1]]), autograd=True)
target = Tensor(np.array([[0],[1],[0],[1]]), autograd=True)

w = list()
w.append(Tensor(np.random.rand(2,3), autograd=True))
w.append(Tensor(np.random.rand(3,1), autograd=True))

optim = SGD(parameters=w, alpha=0.1)

for i in range(10):

    # Predict
    pred = data.mm(w[0]).mm(w[1])

    # Compare
    loss = ((pred - target)*(pred - target)).sum(0)

    # Learn
    loss.backward(Tensor(np.ones_like(loss.data)))
    optim.step()

    print(loss)

Hasilnya sama persis dengan loop sebelumnya, tetapi update bobot sekarang cukup satu
baris `optim.step()`. `step` juga menolkan gradien (`zero=True`) supaya gradien putaran
berikutnya tidak menumpuk di atas gradien sebelumnya.

## 9. Layer

Abstraksi yang paling umum di semua framework adalah layer: kumpulan teknik forward
propagation yang dikemas dengan satu metode `.forward()`. `Layer` adalah kelas dasar
dengan daftar `parameters` dan metode `get_parameters()`. `Linear` adalah layer linear
sungguhan: bobot (diinisialisasi dengan `randn * sqrt(2/n_inputs)`) ditambah bias.

In [ ]:
class Layer(object):

    def __init__(self):
        self.parameters = list()

    def get_parameters(self):
        return self.parameters


class Linear(Layer):

    def __init__(self, n_inputs, n_outputs):
        super().__init__()
        W = np.random.randn(n_inputs, n_outputs) * np.sqrt(2.0/(n_inputs))
        self.weight = Tensor(W, autograd=True)
        self.bias = Tensor(np.zeros(n_outputs), autograd=True)

        self.parameters.append(self.weight)
        self.parameters.append(self.bias)

    def forward(self, input):
        return input.mm(self.weight)+self.bias.expand(0,len(input.data))

`bias.expand(0, len(input.data))` menyalin bias sebanyak jumlah baris input, sehingga
bisa dijumlahkan dengan hasil `mm`. Tidak ada yang baru di sini selain pengorganisasian
bobot ke dalam kelas.

## 10. Layer yang berisi layer

Layer juga bisa berisi layer lain. Yang paling populer adalah `Sequential`, yang
mengalirkan input melewati daftar layer, keluaran satu layer menjadi masukan layer
berikutnya. `get_parameters()` mengumpulkan parameter dari semua layer di dalamnya.

In [ ]:
class Sequential(Layer):

    def __init__(self, layers=list()):
        super().__init__()

        self.layers = layers

    def add(self, layer):
        self.layers.append(layer)

    def forward(self, input):
        for layer in self.layers:
            input = layer.forward(input)
        return input

    def get_parameters(self):
        params = list()
        for l in self.layers:
            params += l.get_parameters()
        return params

import numpy
np.random.seed(0)

data = Tensor(np.array([[0,0],[0,1],[1,0],[1,1]]), autograd=True)
target = Tensor(np.array([[0],[1],[0],[1]]), autograd=True)

model = Sequential([Linear(2,3), Linear(3,1)])

optim = SGD(parameters=model.get_parameters(), alpha=0.05)

for i in range(10):

    # Predict
    pred = model.forward(data)

    # Compare
    loss = ((pred - target)*(pred - target)).sum(0)

    # Learn
    loss.backward(Tensor(np.ones_like(loss.data)))
    optim.step()
    print(loss)

## 11. Layer fungsi loss

Layer juga bisa berupa fungsi tanpa bobot. Contoh paling populer adalah fungsi loss,
misalnya mean squared error.

In [ ]:
class MSELoss(Layer):

    def __init__(self):
        super().__init__()

    def forward(self, pred, target):
        return ((pred - target)*(pred - target)).sum(0)

import numpy
np.random.seed(0)

data = Tensor(np.array([[0,0],[0,1],[1,0],[1,1]]), autograd=True)
target = Tensor(np.array([[0],[1],[0],[1]]), autograd=True)

model = Sequential([Linear(2,3), Linear(3,1)])
criterion = MSELoss()

optim = SGD(parameters=model.get_parameters(), alpha=0.05)

for i in range(10):

    # Predict
    pred = model.forward(data)

    # Compare
    loss = criterion.forward(pred, target)

    # Learn
    loss.backward(Tensor(np.ones_like(loss.data)))
    optim.step()
    print(loss)

Hasil `Sequential` dan versi dengan `MSELoss` sama persis (2.334 turun ke 0.0089).
Memindahkan perhitungan loss ke kelas tidak mengubah hitungan; hanya merapikan kode dan
memastikan urutan langkah forward selalu benar.

### Cara mempelajari sebuah framework

Kalau diringkas, framework adalah autograd ditambah daftar layer dan optimizer yang sudah
jadi. Dengan memandang framework seperti itu, belajar framework baru jadi lebih mudah:
cari contoh kode paling sederhana, ubah sedikit untuk mengenal API autograd-nya, lalu
modifikasi bertahap sampai ke eksperimen yang diinginkan. Framework yang API-nya paling
mirip dengan yang dibangun di sini adalah PyTorch.

## 12. Layer nonlinearitas

Untuk chapter berikutnya, `Tensor` perlu `.sigmoid()` dan `.tanh()`. Turunannya sudah
dipelajari sebelumnya: untuk sigmoid `output * (1 - output)`, dan untuk tanh
`1 - output * output`. Di bagian `backward`, gradien dari atas dikalikan turunan itu.
Setelah itu dibuat layer `Tanh` dan `Sigmoid` yang bisa diselipkan ke `Sequential`.

In [ ]:
import numpy as np

class Tensor (object):

    def __init__(self,data,
                 autograd=False,
                 creators=None,
                 creation_op=None,
                 id=None):

        self.data = np.array(data)
        self.autograd = autograd
        self.grad = None
        if(id is None):
            self.id = np.random.randint(0,100000)
        else:
            self.id = id

        self.creators = creators
        self.creation_op = creation_op
        self.children = {}

        if(creators is not None):
            for c in creators:
                if(self.id not in c.children):
                    c.children[self.id] = 1
                else:
                    c.children[self.id] += 1

    def all_children_grads_accounted_for(self):
        for id,cnt in self.children.items():
            if(cnt != 0):
                return False
        return True

    def backward(self,grad=None, grad_origin=None):
        if(self.autograd):

            if(grad is None):
                grad = Tensor(np.ones_like(self.data))

            if(grad_origin is not None):
                if(self.children[grad_origin.id] == 0):
                    raise Exception("cannot backprop more than once")
                else:
                    self.children[grad_origin.id] -= 1

            if(self.grad is None):
                self.grad = grad
            else:
                self.grad += grad

            # grads must not have grads of their own
            assert grad.autograd == False

            # only continue backpropping if there's something to
            # backprop into and if all gradients (from children)
            # are accounted for override waiting for children if
            # "backprop" was called on this variable directly
            if(self.creators is not None and
               (self.all_children_grads_accounted_for() or
                grad_origin is None)):

                if(self.creation_op == "add"):
                    self.creators[0].backward(self.grad, self)
                    self.creators[1].backward(self.grad, self)

                if(self.creation_op == "sub"):
                    self.creators[0].backward(Tensor(self.grad.data), self)
                    self.creators[1].backward(Tensor(self.grad.__neg__().data), self)

                if(self.creation_op == "mul"):
                    new = self.grad * self.creators[1]
                    self.creators[0].backward(new , self)
                    new = self.grad * self.creators[0]
                    self.creators[1].backward(new, self)

                if(self.creation_op == "mm"):
                    c0 = self.creators[0]
                    c1 = self.creators[1]
                    new = self.grad.mm(c1.transpose())
                    c0.backward(new)
                    new = self.grad.transpose().mm(c0).transpose()
                    c1.backward(new)

                if(self.creation_op == "transpose"):
                    self.creators[0].backward(self.grad.transpose())

                if("sum" in self.creation_op):
                    dim = int(self.creation_op.split("_")[1])
                    self.creators[0].backward(self.grad.expand(dim,
                                                               self.creators[0].data.shape[dim]))

                if("expand" in self.creation_op):
                    dim = int(self.creation_op.split("_")[1])
                    self.creators[0].backward(self.grad.sum(dim))

                if(self.creation_op == "neg"):
                    self.creators[0].backward(self.grad.__neg__())

                if(self.creation_op == "sigmoid"):
                    ones = Tensor(np.ones_like(self.grad.data))
                    self.creators[0].backward(self.grad * (self * (ones - self)))

                if(self.creation_op == "tanh"):
                    ones = Tensor(np.ones_like(self.grad.data))
                    self.creators[0].backward(self.grad * (ones - (self * self)))

    def __add__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data + other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="add")
        return Tensor(self.data + other.data)

    def __neg__(self):
        if(self.autograd):
            return Tensor(self.data * -1,
                          autograd=True,
                          creators=[self],
                          creation_op="neg")
        return Tensor(self.data * -1)

    def __sub__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data - other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="sub")
        return Tensor(self.data - other.data)

    def __mul__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data * other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="mul")
        return Tensor(self.data * other.data)

    def sum(self, dim):
        if(self.autograd):
            return Tensor(self.data.sum(dim),
                          autograd=True,
                          creators=[self],
                          creation_op="sum_"+str(dim))
        return Tensor(self.data.sum(dim))

    def expand(self, dim,copies):

        trans_cmd = list(range(0,len(self.data.shape)))
        trans_cmd.insert(dim,len(self.data.shape))
        new_data = self.data.repeat(copies).reshape(list(self.data.shape) + [copies]).transpose(trans_cmd)

        if(self.autograd):
            return Tensor(new_data,
                          autograd=True,
                          creators=[self],
                          creation_op="expand_"+str(dim))
        return Tensor(new_data)

    def transpose(self):
        if(self.autograd):
            return Tensor(self.data.transpose(),
                          autograd=True,
                          creators=[self],
                          creation_op="transpose")

        return Tensor(self.data.transpose())

    def mm(self, x):
        if(self.autograd):
            return Tensor(self.data.dot(x.data),
                          autograd=True,
                          creators=[self,x],
                          creation_op="mm")
        return Tensor(self.data.dot(x.data))

    def sigmoid(self):
        if(self.autograd):
            return Tensor(1 / (1 + np.exp(-self.data)),
                          autograd=True,
                          creators=[self],
                          creation_op="sigmoid")
        return Tensor(1 / (1 + np.exp(-self.data)))

    def tanh(self):
        if(self.autograd):
            return Tensor(np.tanh(self.data),
                          autograd=True,
                          creators=[self],
                          creation_op="tanh")
        return Tensor(np.tanh(self.data))

    def __repr__(self):
        return str(self.data.__repr__())

    def __str__(self):
        return str(self.data.__str__())

class Tanh(Layer):
    def __init__(self):
        super().__init__()

    def forward(self, input):
        return input.tanh()

class Sigmoid(Layer):
    def __init__(self):
        super().__init__()

    def forward(self, input):
        return input.sigmoid()

In [ ]:
import numpy
np.random.seed(0)

data = Tensor(np.array([[0,0],[0,1],[1,0],[1,1]]), autograd=True)
target = Tensor(np.array([[0],[1],[0],[1]]), autograd=True)

model = Sequential([Linear(2,3), Tanh(), Linear(3,1), Sigmoid()])
criterion = MSELoss()

optim = SGD(parameters=model.get_parameters(), alpha=1)

for i in range(10):

    # Predict
    pred = model.forward(data)

    # Compare
    loss = criterion.forward(pred, target)

    # Learn
    loss.backward(Tensor(np.ones_like(loss.data)))
    optim.step()
    print(loss)

`Tanh()` dan `Sigmoid()` bisa langsung dimasukkan ke `Sequential`, dan jaringan tahu cara
memakainya. Loss turun dari 1.06 ke 0.047 dalam 10 iterasi.

## 13. Embedding layer

Embedding layer menerjemahkan indeks menjadi aktivasi. Seperti di Chapter 11, kalau
kosakatanya 200 kata maka ada 200 embedding, satu baris matriks bobot per kata. Awalnya
yang dibuat hanya matriksnya, dengan inisialisasi kecil acak (kebiasaan dari word2vec).

In [ ]:
class Embedding(Layer):

    def __init__(self, vocab_size, dim):
        super().__init__()

        self.vocab_size = vocab_size
        self.dim = dim

        # this random initialiation style is just a convention from word2vec
        self.weight = (np.random.rand(vocab_size, dim) - 0.5) / dim

Untuk forward propagation, yang dimasukkan adalah indeks kata, bukan katanya. NumPy
mendukung hal ini: jika sebuah matriks diindeks dengan matriks bilangan bulat, hasilnya
adalah matriks yang sama bentuknya dengan tiap bilangan diganti baris yang ditunjuk.
Jadi matriks indeks 2 dimensi berubah menjadi matriks embedding 3 dimensi.

In [ ]:
identity = np.eye(5)
print(identity)

In [ ]:
print(identity[np.array([[1,2,3,4],
                         [2,3,4,0]])])

## 14. Menambah indexing ke autograd

Agar embedding layer bisa dilatih, autograd harus mendukung indexing. Saat
backpropagation, gradien harus ditaruh di baris yang sama dengan baris yang dipilih saat
forward. Karena itu indeks yang dipakai disimpan (`index_select_indices`) supaya bisa
dipakai lagi. Di `backward`, dibuat gradien nol seukuran matriks asli, indeks diratakan
agar bisa di-loop, gradien dijadikan daftar baris, lalu tiap gradien ditambahkan ke baris
yang sesuai. Kalau satu indeks dipakai dua kali, barisnya menerima dua gradien.

In [ ]:
import numpy as np

class Tensor (object):

    def __init__(self,data,
                 autograd=False,
                 creators=None,
                 creation_op=None,
                 id=None):

        self.data = np.array(data)
        self.autograd = autograd
        self.grad = None
        if(id is None):
            self.id = np.random.randint(0,100000)
        else:
            self.id = id

        self.creators = creators
        self.creation_op = creation_op
        self.children = {}

        if(creators is not None):
            for c in creators:
                if(self.id not in c.children):
                    c.children[self.id] = 1
                else:
                    c.children[self.id] += 1

    def all_children_grads_accounted_for(self):
        for id,cnt in self.children.items():
            if(cnt != 0):
                return False
        return True

    def backward(self,grad=None, grad_origin=None):
        if(self.autograd):

            if(grad is None):
                grad = Tensor(np.ones_like(self.data))

            if(grad_origin is not None):
                if(self.children[grad_origin.id] == 0):
                    raise Exception("cannot backprop more than once")
                else:
                    self.children[grad_origin.id] -= 1

            if(self.grad is None):
                self.grad = grad
            else:
                self.grad += grad

            # grads must not have grads of their own
            assert grad.autograd == False

            # only continue backpropping if there's something to
            # backprop into and if all gradients (from children)
            # are accounted for override waiting for children if
            # "backprop" was called on this variable directly
            if(self.creators is not None and
               (self.all_children_grads_accounted_for() or
                grad_origin is None)):

                if(self.creation_op == "add"):
                    self.creators[0].backward(self.grad, self)
                    self.creators[1].backward(self.grad, self)

                if(self.creation_op == "sub"):
                    self.creators[0].backward(Tensor(self.grad.data), self)
                    self.creators[1].backward(Tensor(self.grad.__neg__().data), self)

                if(self.creation_op == "mul"):
                    new = self.grad * self.creators[1]
                    self.creators[0].backward(new , self)
                    new = self.grad * self.creators[0]
                    self.creators[1].backward(new, self)

                if(self.creation_op == "mm"):
                    c0 = self.creators[0]
                    c1 = self.creators[1]
                    new = self.grad.mm(c1.transpose())
                    c0.backward(new)
                    new = self.grad.transpose().mm(c0).transpose()
                    c1.backward(new)

                if(self.creation_op == "transpose"):
                    self.creators[0].backward(self.grad.transpose())

                if("sum" in self.creation_op):
                    dim = int(self.creation_op.split("_")[1])
                    self.creators[0].backward(self.grad.expand(dim,
                                                               self.creators[0].data.shape[dim]))

                if("expand" in self.creation_op):
                    dim = int(self.creation_op.split("_")[1])
                    self.creators[0].backward(self.grad.sum(dim))

                if(self.creation_op == "neg"):
                    self.creators[0].backward(self.grad.__neg__())

                if(self.creation_op == "sigmoid"):
                    ones = Tensor(np.ones_like(self.grad.data))
                    self.creators[0].backward(self.grad * (self * (ones - self)))

                if(self.creation_op == "tanh"):
                    ones = Tensor(np.ones_like(self.grad.data))
                    self.creators[0].backward(self.grad * (ones - (self * self)))

                if(self.creation_op == "index_select"):
                    new_grad = np.zeros_like(self.creators[0].data)
                    indices_ = self.index_select_indices.data.flatten()
                    grad_ = grad.data.reshape(len(indices_), -1)
                    for i in range(len(indices_)):
                        new_grad[indices_[i]] += grad_[i]
                    self.creators[0].backward(Tensor(new_grad))

    def __add__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data + other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="add")
        return Tensor(self.data + other.data)

    def __neg__(self):
        if(self.autograd):
            return Tensor(self.data * -1,
                          autograd=True,
                          creators=[self],
                          creation_op="neg")
        return Tensor(self.data * -1)

    def __sub__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data - other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="sub")
        return Tensor(self.data - other.data)

    def __mul__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data * other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="mul")
        return Tensor(self.data * other.data)

    def sum(self, dim):
        if(self.autograd):
            return Tensor(self.data.sum(dim),
                          autograd=True,
                          creators=[self],
                          creation_op="sum_"+str(dim))
        return Tensor(self.data.sum(dim))

    def expand(self, dim,copies):

        trans_cmd = list(range(0,len(self.data.shape)))
        trans_cmd.insert(dim,len(self.data.shape))
        new_data = self.data.repeat(copies).reshape(list(self.data.shape) + [copies]).transpose(trans_cmd)

        if(self.autograd):
            return Tensor(new_data,
                          autograd=True,
                          creators=[self],
                          creation_op="expand_"+str(dim))
        return Tensor(new_data)

    def transpose(self):
        if(self.autograd):
            return Tensor(self.data.transpose(),
                          autograd=True,
                          creators=[self],
                          creation_op="transpose")

        return Tensor(self.data.transpose())

    def mm(self, x):
        if(self.autograd):
            return Tensor(self.data.dot(x.data),
                          autograd=True,
                          creators=[self,x],
                          creation_op="mm")
        return Tensor(self.data.dot(x.data))

    def sigmoid(self):
        if(self.autograd):
            return Tensor(1 / (1 + np.exp(-self.data)),
                          autograd=True,
                          creators=[self],
                          creation_op="sigmoid")
        return Tensor(1 / (1 + np.exp(-self.data)))

    def tanh(self):
        if(self.autograd):
            return Tensor(np.tanh(self.data),
                          autograd=True,
                          creators=[self],
                          creation_op="tanh")
        return Tensor(np.tanh(self.data))

    def index_select(self, indices):

        if(self.autograd):
            new = Tensor(self.data[indices.data],
                         autograd=True,
                         creators=[self],
                         creation_op="index_select")
            new.index_select_indices = indices
            return new
        return Tensor(self.data[indices.data])

    def __repr__(self):
        return str(self.data.__repr__())

    def __str__(self):
        return str(self.data.__str__())

x = Tensor(np.eye(5), autograd=True)
x.index_select(Tensor([[1,2,3],[2,3,4]])).backward()
print(x.grad)

Baris 0 bernilai nol karena tidak pernah dipilih, baris 1 dan 4 menerima gradien satu kali,
dan baris 2 dan 3 menerima dua kali karena indeks 2 dan 3 muncul dua kali. Jadi gradien
dikembalikan ke baris yang tepat dan terakumulasi sesuai jumlah pemakaian.

## 15. Embedding layer (revisi)

Sekarang forward propagation cukup memanggil `.index_select()`, dan autograd mengurus
sisanya. Bobot embedding dibuat sebagai `Tensor` dengan `autograd=True` dan dimasukkan
ke daftar `parameters`.

In [ ]:
class Embedding(Layer):

    def __init__(self, vocab_size, dim):
        super().__init__()

        self.vocab_size = vocab_size
        self.dim = dim

        # this random initialiation style is just a convention from word2vec
        self.weight = Tensor((np.random.rand(vocab_size, dim) - 0.5) / dim, autograd=True)

        self.parameters.append(self.weight)

    def forward(self, input):
        return self.weight.index_select(input)

In [ ]:
import numpy
np.random.seed(0)

data = Tensor(np.array([1,2,1,2]), autograd=True)
target = Tensor(np.array([[0],[1],[0],[1]]), autograd=True)

embed = Embedding(5,3)
model = Sequential([embed, Tanh(), Linear(3,1), Sigmoid()])
criterion = MSELoss()

optim = SGD(parameters=model.get_parameters(), alpha=0.5)

for i in range(10):

    # Predict
    pred = model.forward(data)

    # Compare
    loss = criterion.forward(pred, target)

    # Learn
    loss.backward(Tensor(np.ones_like(loss.data)))
    optim.step()
    print(loss)

Jaringan ini belajar mengaitkan indeks input 1 dan 2 dengan prediksi 0 dan 1. Indeks
itu nantinya bisa mewakili kata; contoh ini hanya untuk membuktikan embedding bekerja.

## 16. Layer cross entropy

Cross entropy menghitung softmax dan loss sekaligus. `cross_entropy` di `Tensor`
menghitung softmax output, distribusi target (one-hot), lalu loss negative log
likelihood rata-rata. Gradiennya sangat sederhana: `softmax_output - target_dist`.

In [ ]:
import numpy as np

class Tensor (object):

    def __init__(self,data,
                 autograd=False,
                 creators=None,
                 creation_op=None,
                 id=None):

        self.data = np.array(data)
        self.autograd = autograd
        self.grad = None
        if(id is None):
            self.id = np.random.randint(0,100000)
        else:
            self.id = id

        self.creators = creators
        self.creation_op = creation_op
        self.children = {}

        if(creators is not None):
            for c in creators:
                if(self.id not in c.children):
                    c.children[self.id] = 1
                else:
                    c.children[self.id] += 1

    def all_children_grads_accounted_for(self):
        for id,cnt in self.children.items():
            if(cnt != 0):
                return False
        return True

    def backward(self,grad=None, grad_origin=None):
        if(self.autograd):

            if(grad is None):
                grad = Tensor(np.ones_like(self.data))

            if(grad_origin is not None):
                if(self.children[grad_origin.id] == 0):
                    raise Exception("cannot backprop more than once")
                else:
                    self.children[grad_origin.id] -= 1

            if(self.grad is None):
                self.grad = grad
            else:
                self.grad += grad

            # grads must not have grads of their own
            assert grad.autograd == False

            # only continue backpropping if there's something to
            # backprop into and if all gradients (from children)
            # are accounted for override waiting for children if
            # "backprop" was called on this variable directly
            if(self.creators is not None and
               (self.all_children_grads_accounted_for() or
                grad_origin is None)):

                if(self.creation_op == "add"):
                    self.creators[0].backward(self.grad, self)
                    self.creators[1].backward(self.grad, self)

                if(self.creation_op == "sub"):
                    self.creators[0].backward(Tensor(self.grad.data), self)
                    self.creators[1].backward(Tensor(self.grad.__neg__().data), self)

                if(self.creation_op == "mul"):
                    new = self.grad * self.creators[1]
                    self.creators[0].backward(new , self)
                    new = self.grad * self.creators[0]
                    self.creators[1].backward(new, self)

                if(self.creation_op == "mm"):
                    c0 = self.creators[0]
                    c1 = self.creators[1]
                    new = self.grad.mm(c1.transpose())
                    c0.backward(new)
                    new = self.grad.transpose().mm(c0).transpose()
                    c1.backward(new)

                if(self.creation_op == "transpose"):
                    self.creators[0].backward(self.grad.transpose())

                if("sum" in self.creation_op):
                    dim = int(self.creation_op.split("_")[1])
                    self.creators[0].backward(self.grad.expand(dim,
                                                               self.creators[0].data.shape[dim]))

                if("expand" in self.creation_op):
                    dim = int(self.creation_op.split("_")[1])
                    self.creators[0].backward(self.grad.sum(dim))

                if(self.creation_op == "neg"):
                    self.creators[0].backward(self.grad.__neg__())

                if(self.creation_op == "sigmoid"):
                    ones = Tensor(np.ones_like(self.grad.data))
                    self.creators[0].backward(self.grad * (self * (ones - self)))

                if(self.creation_op == "tanh"):
                    ones = Tensor(np.ones_like(self.grad.data))
                    self.creators[0].backward(self.grad * (ones - (self * self)))

                if(self.creation_op == "index_select"):
                    new_grad = np.zeros_like(self.creators[0].data)
                    indices_ = self.index_select_indices.data.flatten()
                    grad_ = grad.data.reshape(len(indices_), -1)
                    for i in range(len(indices_)):
                        new_grad[indices_[i]] += grad_[i]
                    self.creators[0].backward(Tensor(new_grad))

                if(self.creation_op == "cross_entropy"):
                    dx = self.softmax_output - self.target_dist
                    self.creators[0].backward(Tensor(dx))

    def __add__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data + other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="add")
        return Tensor(self.data + other.data)

    def __neg__(self):
        if(self.autograd):
            return Tensor(self.data * -1,
                          autograd=True,
                          creators=[self],
                          creation_op="neg")
        return Tensor(self.data * -1)

    def __sub__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data - other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="sub")
        return Tensor(self.data - other.data)

    def __mul__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data * other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="mul")
        return Tensor(self.data * other.data)

    def sum(self, dim):
        if(self.autograd):
            return Tensor(self.data.sum(dim),
                          autograd=True,
                          creators=[self],
                          creation_op="sum_"+str(dim))
        return Tensor(self.data.sum(dim))

    def expand(self, dim,copies):

        trans_cmd = list(range(0,len(self.data.shape)))
        trans_cmd.insert(dim,len(self.data.shape))
        new_data = self.data.repeat(copies).reshape(list(self.data.shape) + [copies]).transpose(trans_cmd)

        if(self.autograd):
            return Tensor(new_data,
                          autograd=True,
                          creators=[self],
                          creation_op="expand_"+str(dim))
        return Tensor(new_data)

    def transpose(self):
        if(self.autograd):
            return Tensor(self.data.transpose(),
                          autograd=True,
                          creators=[self],
                          creation_op="transpose")

        return Tensor(self.data.transpose())

    def mm(self, x):
        if(self.autograd):
            return Tensor(self.data.dot(x.data),
                          autograd=True,
                          creators=[self,x],
                          creation_op="mm")
        return Tensor(self.data.dot(x.data))

    def sigmoid(self):
        if(self.autograd):
            return Tensor(1 / (1 + np.exp(-self.data)),
                          autograd=True,
                          creators=[self],
                          creation_op="sigmoid")
        return Tensor(1 / (1 + np.exp(-self.data)))

    def tanh(self):
        if(self.autograd):
            return Tensor(np.tanh(self.data),
                          autograd=True,
                          creators=[self],
                          creation_op="tanh")
        return Tensor(np.tanh(self.data))

    def index_select(self, indices):

        if(self.autograd):
            new = Tensor(self.data[indices.data],
                         autograd=True,
                         creators=[self],
                         creation_op="index_select")
            new.index_select_indices = indices
            return new
        return Tensor(self.data[indices.data])

    def cross_entropy(self, target_indices):

        temp = np.exp(self.data)
        softmax_output = temp / np.sum(temp,
                                       axis=len(self.data.shape)-1,
                                       keepdims=True)

        t = target_indices.data.flatten()
        p = softmax_output.reshape(len(t),-1)
        target_dist = np.eye(p.shape[1])[t]
        loss = -(np.log(p) * (target_dist)).sum(1).mean()

        if(self.autograd):
            out = Tensor(loss,
                         autograd=True,
                         creators=[self],
                         creation_op="cross_entropy")
            out.softmax_output = softmax_output
            out.target_dist = target_dist
            return out

        return Tensor(loss)

    def __repr__(self):
        return str(self.data.__repr__())

    def __str__(self):
        return str(self.data.__str__())

In [ ]:
class CrossEntropyLoss(object):

    def __init__(self):
        super().__init__()

    def forward(self, input, target):
        return input.cross_entropy(target)

In [ ]:
import numpy
np.random.seed(0)

# data indices
data = Tensor(np.array([1,2,1,2]), autograd=True)

# target indices
target = Tensor(np.array([0,1,0,1]), autograd=True)

model = Sequential([Embedding(3,3), Tanh(), Linear(3,4)])
criterion = CrossEntropyLoss()

optim = SGD(parameters=model.get_parameters(), alpha=0.1)

for i in range(10):

    # Predict
    pred = model.forward(data)

    # Compare
    loss = criterion.forward(pred, target)

    # Learn
    loss.backward(Tensor(np.ones_like(loss.data)))
    optim.step()
    print(loss)

Loss turun dari 1.39 ke 0.165 dalam 10 iterasi.

Satu hal yang berbeda dari loss lain: softmax dan perhitungan loss berada di dalam kelas
loss yang sama. Ini konvensi yang sangat umum. Untuk melatih dengan cross entropy,
softmax dibuang dari forward propagation, dan kelas cross entropy menjalankan softmax
sebagai bagian dari loss. Alasannya performa: menghitung gradien softmax dan negative
log likelihood sekaligus lebih cepat daripada dua modul terpisah, karena ada jalan
pintas dalam matematika gradiennya (yaitu `softmax_output - target_dist` tadi).

## 17. Recurrent neural network layer

Latihan terakhir: menyusun satu layer dari beberapa layer kecil. `RNNCell` dibuat dari
tiga layer linear:

- `w_ih`: input ke hidden (memproses data masukan),
- `w_hh`: hidden ke hidden (memperbarui hidden berdasarkan hidden sebelumnya),
- `w_ho`: hidden ke output (prediksi dari vektor hidden).

`forward` menerima input saat ini dan hidden sebelumnya, lalu mengembalikan output dan
hidden baru. State yang diteruskan antar langkah waktu adalah `hidden`. Parameter
`activation` memilih nonlinearitas yang dipakai pada hidden.

In [ ]:
class RNNCell(Layer):

    def __init__(self, n_inputs, n_hidden, n_output, activation='sigmoid'):
        super().__init__()

        self.n_inputs = n_inputs
        self.n_hidden = n_hidden
        self.n_output = n_output

        if(activation == 'sigmoid'):
            self.activation = Sigmoid()
        elif(activation == 'tanh'):
            self.activation == Tanh()
        else:
            raise Exception("Non-linearity not found")

        self.w_ih = Linear(n_inputs, n_hidden)
        self.w_hh = Linear(n_hidden, n_hidden)
        self.w_ho = Linear(n_hidden, n_output)

        self.parameters += self.w_ih.get_parameters()
        self.parameters += self.w_hh.get_parameters()
        self.parameters += self.w_ho.get_parameters()

    def forward(self, input, hidden):
        from_prev_hidden = self.w_hh.forward(hidden)
        combined = self.w_ih.forward(input) + from_prev_hidden
        new_hidden = self.activation.forward(combined)
        output = self.w_ho.forward(new_hidden)
        return output, new_hidden

    def init_hidden(self, batch_size=1):
        return Tensor(np.zeros((batch_size,self.n_hidden)), autograd=True)

Satu hal yang saya temukan di kode ini: pada cabang `'tanh'`, tertulis
`self.activation == Tanh()` (dua tanda sama dengan), bukan `=`. Saat saya coba
`RNNCell(..., activation='tanh')`, langsung muncul `AttributeError` karena
`self.activation` belum ada. Kodenya tetap saya tulis apa adanya, dan hanya
`'sigmoid'` (nilai bawaan) yang dipakai di bawah.

Sekarang jaringan ini dipakai pada tugas yang sama dengan akhir Chapter 12, yaitu
memprediksi kata berikutnya pada dataset bAbI. Data disiapkan dengan cara yang sama,
dengan satu tambahan: tiap kalimat dipadatkan dengan `-` di depan sampai panjangnya 6,
agar semua baris sama panjang dan bisa diproses sebagai batch. Vocabulary diurutkan
(`sorted`) agar hasilnya sama setiap dijalankan, sedangkan versi `list(set(...))`
urutannya bisa berubah antar sesi.

In [ ]:
import os, urllib.request

RAW = 'https://raw.githubusercontent.com/iamtrask/Grokking-Deep-Learning/{}/{}'

def ensure_file(name):
    if os.path.exists(name):
        return
    d = os.path.dirname(name)
    if d:
        os.makedirs(d, exist_ok=True)
    for branch in ['master', 'main']:          # cabang repo bisa master atau main
        try:
            urllib.request.urlretrieve(RAW.format(branch, name), name)
            return
        except Exception:
            pass
    raise RuntimeError('Gagal mengunduh ' + name)

ensure_file('tasksv11/en/qa1_single-supporting-fact_train.txt')

import sys,random,math
from collections import Counter
import numpy as np

f = open('tasksv11/en/qa1_single-supporting-fact_train.txt','r')
raw = f.readlines()
f.close()

tokens = list()
for line in raw[0:1000]:
    tokens.append(line.lower().replace("\n","").split(" ")[1:])

new_tokens = list()
for line in tokens:
    new_tokens.append(['-'] * (6 - len(line)) + line)

tokens = new_tokens

vocab = set()
for sent in tokens:
    for word in sent:
        vocab.add(word)

vocab = sorted(vocab)

word2index = {}
for i,word in enumerate(vocab):
    word2index[word]=i

def words2indices(sentence):
    idx = list()
    for word in sentence:
        idx.append(word2index[word])
    return idx

indices = list()
for line in tokens:
    idx = list()
    for w in line:
        idx.append(word2index[w])
    indices.append(idx)

data = np.array(indices)

print('ukuran vocabulary:', len(vocab))
print('bentuk data      :', data.shape)
print(tokens[2])

Berikutnya embedding dan recurrent cell dibuat, dengan cross entropy sebagai loss dan
SGD sebagai optimizer. Perhatikan bahwa jaringan ini sedikit lebih kompleks daripada
yang di Chapter 12 (ada satu layer tambahan), tetapi kodenya jauh lebih ringkas
berkat framework mini ini. Istilah *cell* adalah nama konvensional untuk layer rekuren
yang mengimplementasikan satu langkah rekurensi saja. Layer yang menyusun banyak cell
akan disebut RNN dengan parameter `n_layers`.

In [ ]:
embed = Embedding(vocab_size=len(vocab),dim=16)
model = RNNCell(n_inputs=16, n_hidden=16, n_output=len(vocab))

criterion = CrossEntropyLoss()
optim = SGD(parameters=model.get_parameters() + embed.get_parameters(), alpha=0.05)

Loop training mengambil 100 contoh pertama sebagai batch. Pada setiap iterasi, jaringan
membaca lima kata (`t` dari 0 sampai 4) sambil meneruskan `hidden`, lalu memprediksi kata
keenam (`data[0:batch_size, t+1]` dengan `t = 4`). `loss.backward()` tanpa argumen
memakai gradien berisi 1 yang tadi ditambahkan sebagai kemudahan.

In [ ]:
for iter in range(1000):
    batch_size = 100
    total_loss = 0

    hidden = model.init_hidden(batch_size=batch_size)

    for t in range(5):
        input = Tensor(data[0:batch_size,t], autograd=True)
        rnn_input = embed.forward(input=input)
        output, hidden = model.forward(input=rnn_input, hidden=hidden)

    target = Tensor(data[0:batch_size,t+1], autograd=True)
    loss = criterion.forward(output, target)
    loss.backward()
    optim.step()
    total_loss += loss.data
    if(iter % 200 == 0):
        p_correct = (target.data == np.argmax(output.data,axis=1)).mean()
        print("Loss:",total_loss / (len(data)/batch_size),"% Correct:",p_correct)

Akurasi pada 100 contoh pertama naik dari 0.0 pada iterasi 0 ke sekitar 0.36 pada
iterasi 800, dan loss yang dicetak turun dari 0.49 ke 0.136. Angka akurasi ini tidak
mendekati 100% wajar, karena lima kata sebelumnya sering belum cukup untuk mengetahui
tempat yang benar (sama seperti keterbatasan di akhir Chapter 12).

Satu catatan tentang angka loss: yang dicetak adalah `total_loss / (len(data)/batch_size)`,
yaitu dibagi 10, padahal tiap iterasi hanya satu batch yang dihitung. Jadi nilainya
sepersepuluh dari loss batch sebenarnya. Tren turunnya tetap valid.

Terakhir, saya lihat prediksi untuk satu contoh (batch size 1):

In [ ]:
batch_size = 1
hidden = model.init_hidden(batch_size=batch_size)
for t in range(5):
    input = Tensor(data[0:batch_size,t], autograd=True)
    rnn_input = embed.forward(input=input)
    output, hidden = model.forward(input=rnn_input, hidden=hidden)

target = Tensor(data[0:batch_size,t+1], autograd=True)
loss = criterion.forward(output, target)

ctx = ""
for idx in data[0:batch_size][0][0:-1]:
    ctx += vocab[idx] + " "
print("Context:",ctx)
print("True:",vocab[target.data[0]])
print("Pred:", vocab[output.data.argmax()])

Jaringan menebak `office.` padahal jawaban yang benar `bathroom.`. Kesalahannya masuk
akal: tebakannya nama tempat, hanya tempatnya yang salah. Konteks `- mary moved to the`
memang tidak memuat informasi tempat tujuan. Dibanding Chapter 12, kode jaringannya
jauh lebih singkat karena tidak ada lagi gradien yang ditulis tangan.

## Key Ideas — Chapter 13

- Framework deep learning pada intinya adalah autograd ditambah daftar layer dan optimizer.
- Tensor menyimpan data, `creators`, dan `creation_op`, sehingga operasi membangun computation graph secara otomatis (dynamic graph).
- Gradien dihitung dengan memanggil `.backward()` secara rekursif ke pembuat tensor.
- Tensor yang dipakai berkali-kali harus mengakumulasi gradien dari semua anaknya dan menunggu semuanya sebelum meneruskan gradien.
- Menambah operasi baru hanya butuh dua hal: fungsi operasinya dan blok gradiennya di `.backward()`.
- Dengan autograd, backpropagation tidak perlu ditulis tangan, dan SGD cuma loop pembaruan bobot.
- Layer (Linear, Sequential, loss, Tanh, Sigmoid, Embedding, RNNCell) hanyalah pengemasan forward propagation dengan metode `.forward()` dan `get_parameters()`.
- Softmax dan loss dijadikan satu pada cross entropy karena gradiennya lebih sederhana dan cepat.
- Embedding layer memerlukan indexing di autograd: gradien dikembalikan ke baris yang sama dengan baris yang dipilih.

## Kesimpulan

Chapter 13 membangun framework deep learning mini dari NumPy: `Tensor` dengan autograd,
optimizer SGD, dan kumpulan layer. Tidak ada teori baru; yang dikerjakan adalah
rekayasa agar backpropagation yang selama ini ditulis tangan terjadi otomatis. Hasilnya,
jaringan yang di chapter sebelumnya butuh banyak kode gradien, termasuk RNN, bisa
ditulis hanya dengan forward propagation. Framework ini juga menjadi dasar untuk chapter
berikutnya, yaitu LSTM, yang kodenya akan terlalu rumit jika gradiennya ditulis tangan.